# The State of the Antarctic Ice Shelves Data Linkages Notebook

This notebook is a demonstrator from the **Digital Twin Component for Ice Sheets (DTC-IS)**, an ESA-funded, EO-driven digital twin project.

DTC-IS integrates Earth Observation, in-situ data, reanalysis products, and advanced data-driven models to provide a dynamic and science-based representation of the Greenland and Antarctic ice sheets, including their ice shelves. It supports the monitoring, understanding, and prediction of ice-sheet change, delivering actionable information for research, policy, and scenario-based exploration.

[Project Website](https://dtc-ice-sheets.org/) · [Interactive Dashboard](https://dashboards.dtc-ice-sheets.org/) · [Query API](https://query.dtc-ice-sheets.org)

---

Here we focus on the state and fate of the Antarctic ice shelves use case of the DTC-IS, demonstrating how to use the DTC-IS data linkage module to investigate potentially significant relationships between ice shelf variables.

**Key features**:
- **Authentication:** Guide you through authenticating with the *DTC Query API* using your personal access token.
- **Ice-shelf Data Linkage Analysis:** Demonstrate how to build a small Jupyter Notebook-based application to access and run ice-shelf data linkage analyses using the *DTC Query API*.
- **Extraction of Time Series Data**: Demonstrate how to extract time series data from the DTC.

**Notebook preparations:**
To start, we need to import several libraries and pre-prepared helper functions. These will allow us to use the DTC-IS modules in this notebook to run the data linkage analysis and extract other useful information about the analysis being conducted.

In [ ]:
import sys

print("⏳ Installing dependencies...")
if "google.colab" in sys.modules:
    %pip install -q xarray zarr cartopy s3fs==2025.3.0 ipywidgets==7.7.1 contextily dtc-query-client
    # Ensure the newest version of the helpers package is installed
    %pip uninstall -y dtc_is_notebooks
    %pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir  git+https://github.com/DTC-Ice-Sheets/dtc_is_notebooks.git

In [ ]:
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.io as pio
import shapely
from dtc_query_client import CovariateAnalysisType, IceShelfExtent, StateAndFateApi

from dtc_is_notebook_helpers.uc3_helper_functions import (
    build_covariate_analysis_input_selector,
    extract_timeseries_data,
    get_client,
    plot_covariate_analysis,
    plot_timeseries_data,
    run_data_linkage_analysis,
    widget_credentials_make,
)

pio.renderers.default = "colab"

### Authentication:
First, we need to authenticate with the DTC-IS system to interact with it. To do this, please navigate to the following [webpage](https://query.dtc-ice-sheets.org/auth/get-token) to generate an API token and then enter it below where requested. 

This link directs you to the authentication page of the DTC-IS dashboard. In addition to your unique API key, it contains other useful information, such as usage quotas for the DTC-IS infrastructure. These are only applied when using the compute services provided, and you can monitor your usage using the same link that you used to generate your access token. ***Important:*** Keep your API token secure!

In [ ]:
widget_credentials = widget_credentials_make()

In [ ]:
client = get_client(widget_credentials)

The authenticated client object will now be passed to any DTC-Query-Client calls we make. When you reload this page, you will need to reauthenticate the client using your API key.

### Datasets
There are a large number of datasets available for use within the digital twin, including EO, reanalysis, and in-situ data. Over the ice shelves, the following is available:
- **Ice velocity:** Sentinel-1 yearly/monthly and ITSLive Yearly
- **Subglacial Lakes** *(Thwaites only)*
- **Basal Melt Rate:** Polar Ice Shelves, ITSLive quarterly
- **Sea Ice:** C3S Monthly Concentration and Edge
- **Ocean Reanalysis:** GLORY
- **Ice Shelf Mass Balance Components:** Davison *et al.* (2021)
- **Thwaites subglacial lake discharge:** Gourmelen *et al.* (2025)  *(Thwaites only)* - Ocean moorings 0.8C Isotherm, Thwaites and Pine Island Basal Melt Anomalies, Subglacial Lake Discharge.
- **Surface Mass Balance:** RACMO
- **Surface Melt Maps:** Picard *et al.* (2006)
- **BAS Southern Ocean Moorings**  *(Not available in this analysis)*
- **Calving Front:** 4D Antarctica  *(Not available in this analysis as only spatial)*
- **Topography:** Bedmachine V3, REMA 1000/100m mosaic and 100m gapless *(Not available in this analysis as only spatial)*
- **Atmospheric:** Southern Annular Mode Index, Antarctic Oscillation Index  *(Not available in this analysis as only spatial)*

There are too many variables within these datasets to list here. You are encouraged to explore further to see what the twin can offer. Some of these datasets are not available in this notebook due to not have a temporal dimension

### Ice-shelf Data Linkage Analysis:
Below, we will build a tool to evaluate data linkages over ice shelves. You can choose an ice shelf, a set of variables that interest you, the time range of interest and an analysis type. In this example, please select **Thwaites Ice Shelf**, the following four datasets and the associated variables and a **Correlation** analysis from the input selector in the cell below.

- Ocean Moorings (Gouremelen et al. 2025) -> 0_8_degree_c_isotherm_depth
- Basal Melt Anomalies - Pine Island (Gouremelen et al. 2025) -> Basal Melt Anomalies
- Basal Melt Anomalies - Thwaites (Gouremelen et al. 2025) -> Basal Melt Anomalies
- Subglacial Lake Discharge (Gouremelen et al. 2025) -> Lake Discharge

You can leave the start and end datetimes as they are currently. The DTC-IS modules will automatically select the maximum temporal extent of the time series within these time bounds.

In [ ]:
input_selector = await build_covariate_analysis_input_selector(client)

We are now ready to run the data linkage analysis. The data linkage analysis conducts the following processing steps:
1. Selected data variables are downloaded over the ice shelf extent
2. Data variables are aggregated into time series across the spatial extent using the mean.
3. Data variables are preprocessed so that they cover the same temporal bounds and have the same sample frequency.
4. A data linkage analysis is then conducted with the selected variables in this case using the correlation.

In [ ]:
results = await run_data_linkage_analysis(client=client, input_selector=input_selector)

The results of the data linkage analysis are returned as JSON that we can then parse and plot as joint plot:

In [ ]:
fig = plot_covariate_analysis(plot_data=results, analysis_type=CovariateAnalysisType("corr"))
fig.show(renderer="colab")

We can see that, as expected, from Gourmelen *et al.* (2025), there is a significant correlation between the subglacial lake drainage event in 2014 and the basal melt anomalies over the Thwaites ice shelf. While no significant correlations are detected between the other variables.

**Visualising the Spatial Extents Queried:**
To understand the spatial extent being queried during the analysis, it is also possible to extract the polygons used. Visualising spatial extents on a map is easy with the *DTC Query Client*. 

*Note: if you get an `AttributeError: 'NoneType' object has no attribute 'shape'` then please disable the `cx.basemap(ax, crs=outline_is.crs)` line in the codeblock below, as this is a contextily error*.

In [ ]:
geometry_response = await StateAndFateApi(client).get_geometry(
    ice_shelf_id=input_selector.children[0].children[0].children[1].value, extent=IceShelfExtent("ice_shelf")
)
geometry = shapely.from_geojson(geometry_response.to_json())
outline_is = gpd.GeoDataFrame(
    {"Name": [input_selector.children[0].children[0].children[1].value], "geometry": [geometry]}, crs="epsg:4326"
).to_crs("epsg:3031")

geometry_response = await StateAndFateApi(client).get_geometry(
    ice_shelf_id=input_selector.children[0].children[0].children[1].value, extent=IceShelfExtent("ocean")
)
geometry = shapely.from_geojson(geometry_response.to_json())
outline_ocean = gpd.GeoDataFrame(
    {"Name": [input_selector.children[0].children[0].children[1].value], "geometry": [geometry]}, crs="epsg:4326"
).to_crs("epsg:3031")

ax = outline_ocean.plot(edgecolor="b", alpha=0.25, color="blue")
outline_is.plot(edgecolor="k", alpha=0.5, ax=ax, color="grey")
cx.add_basemap(ax, crs=outline_is.crs)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

**Working with the Time Series:**
We can also extract the Time Series for the variables from the DTC-IS. These can be used either to visualise the data, as we will do here, or for your own analysis outside the digital twin. 

Please note this function still uses the variables that you selected in the ***input selector widget*** defined earlier in the notebook. It sequentially passes each variable through the processing chain and aggregates it to a single time series. By processing it this way, the time series returned may have different temporal extents and sample frequencies.

In [ ]:
timeseries_data = await extract_timeseries_data(client=client, input_selector=input_selector)

The ***timeseries_data*** dictionary contains the time series stored as Pandas Series objects. We can work with these as we would any Pandas object. Here we will plot them, but it is also possible to save them for further analysis.

We have used plotly here to visualise the data. This is a nice visualisation library as it allows for interaction with plots (you can zoom and drag). Please feel free to customise these plots or even to use a visualisation library that you are more comfortable with, such as *matplotlib*.

In [ ]:
fig = plot_timeseries_data(timeseries_data=timeseries_data, pretty_labels=True)
fig.show(renderer="colab")

### Conclusions

Thank you for exploring this DTC-IS demonstrator notebook. We hope it's given you a useful introduction to the state and fate of the Antarctic ice shelves use case, and to working with the DTC-IS framework — feel free to revisit it with different variables and parameters to explore further.

We welcome your feedback, questions, and suggestions for new features. You can reach the DTC-IS team at [support@dtc-ice-sheets.org](mailto:support@dtc-ice-sheets.org).

### References
Davison, B. J., Hogg, A. E., Gourmelen, N., Jakob, L., Wuite, J., Nagler, T., ... & Engdahl, M. E. (2023). Annual mass budget of Antarctic ice shelves from 1997 to 2021. Science Advances, 9(41), eadi0186.

Gourmelen, N., Jakob, L., Holland, P. R., Dutrieux, P., Goldberg, D., Bevan, S., ... & Malczyk, G. (2025). The influence of subglacial lake discharge on Thwaites Glacier ice-shelf melting and grounding-line retreat. Nature Communications, 16(1), 2272.

Picard, G., & Fily, M. (2006). Surface melting observations in Antarctica by microwave radiometers: Correcting 26-year time series from changes in acquisition hours. Remote sensing of environment, 104(3), 325-336.
